<a href="https://colab.research.google.com/github/busycaesar/Attention_Attention_Everywhere/blob/Master/QKV-new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

## Get Model and Tokenizer

In [2]:
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2Model.from_pretrained('gpt2', output_attentions=True)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

## Sentence

In [18]:
sentence = 'The dog did not cross the road because it was'

inputs = tokenizer(sentence, return_tensors='pt')

print(inputs)

{'input_ids': tensor([[ 464, 3290,  750,  407, 3272,  262, 2975,  780,  340,  373]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


## `Slide` Where does this ranking come from?

In [28]:
with torch.no_grad():
    h = model(**inputs, output_hidden_states=True).hidden_states[-1][0, -1]

print(f'Final vector: [{", ".join(f"{x:+.3f}" for x in h[:8].tolist())}, ...]   ({len(h)} numbers)\n')

top_n_tokens = 100

logits = h @ model.wte.weight.T
vocab_size = model.config.vocab_size
probs  = torch.softmax(logits, dim=-1)
top    = torch.topk(probs, top_n_tokens)

print(f'Scored against all {vocab_size} tokens. top {top_n_tokens}:\n')
for p, i in zip(top.values, top.indices):
    print(f'  {tokenizer.decode([i]):<14} {p*100:5.5f}%')

Final vector: [-0.034, +0.005, -0.586, -0.430, +0.343, -0.048, -1.473, +0.525, ...]   (768 numbers)

Scored against all 50257 tokens. top 100:

   afraid        7.08241%
   not           6.68700%
   too           6.57735%
   scared        5.02134%
   in            2.55453%
   a             2.08128%
   frightened    1.79524%
   running       1.64846%
   so            1.63166%
   trying        1.51279%
   on            1.43290%
   "             1.40881%
   being         1.23015%
   fearful       1.20079%
   worried       1.12296%
   barking       0.87016%
   concerned     0.83490%
   very          0.70275%
   out           0.68660%
   still         0.67822%
   chasing       0.64512%
   unable        0.64239%
   already       0.60128%
   going         0.59668%
   under         0.59285%
   stuck         0.59253%
   injured       0.54569%
   just          0.50219%
   caught        0.46798%
   terrified     0.45203%
   moving        0.44456%
   at            0.42157%
   safe          0.41566

In [14]:
tokens = [tokenizer.decode([tid]) for tid in inputs['input_ids'][0]]

it_idx = tokens.index(' it')
dog_idx = tokens.index(' dog')
road_idx = tokens.index(' road')

print(it_idx)
print(dog_idx)
print(road_idx)

8
1
6


In [6]:
print('GPT-2 loaded.')
print(f'Layers : {model.config.n_layer}')
print(f'Heads  : {model.config.n_head}')
print(f'd_model: {model.config.n_embd}')
print(f'd_k    : {model.config.n_embd // model.config.n_head}  (d_model / n_heads)')

GPT-2 loaded.
Layers : 12
Heads  : 12
d_model: 768
d_k    : 64  (d_model / n_heads)


In [7]:
import torch

qkv_store = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        with torch.no_grad():
            x   = input[0]
            qkv = module.c_attn(x)
            d   = model.config.n_embd
            Q, K, V = qkv.split(d, dim=2)
            qkv_store[layer_idx] = (
                Q.squeeze(0).detach(),
                K.squeeze(0).detach(),
                V.squeeze(0).detach()
            )
    return hook

blocks = model.transformer.h if hasattr(model, 'transformer') else model.h

handles = [blocks[L].attn.register_forward_hook(make_hook(L)) for L in range(len(blocks))]

with torch.no_grad():
    outputs = model(**inputs)

for h in handles:
    h.remove()

In [8]:
def fmt(v, n=4):
    return '[' + ', '.join(f'{x:+.3f}' for x in v[:n].tolist()) + ', ...]'

def fmt_ends(v, n=3):
    h = ', '.join(f'{x:+.3f}' for x in v[:n].tolist())
    t = ', '.join(f'{x:+.3f}' for x in v[-n:].tolist())
    return f'[{h}, ... , {t}]   (len {len(v)})'

with torch.no_grad():
    hs = model(**inputs, output_hidden_states=True).hidden_states

n_layers = model.config.n_layer
n_heads  = model.config.n_head
d_head   = model.config.n_embd // n_heads
prior    = list(range(it_idx + 1))
names    = [tokens[i].strip() for i in prior]

print('='*100)
print('Starting vector for "it"  (token embedding + positional encoding)')
print('='*100)
print(f'  x_it = {fmt(hs[0][0, it_idx], 10)}   (768 numbers total)')
print('\n  note: each layer also has an MLP sublayer that adds to x_it.')
print('        the tables below trace the attention half only.\n')

for L in range(n_layers):
    print('\n' + '='*100)
    print(f'LAYER {L}')
    print('='*100)

    hdr = f'{"Head":<6}' + ''.join(f'{n:>10}' for n in names)
    print(hdr); print('-'*len(hdr))
    for H in range(n_heads):
        w = outputs.attentions[L][0][H][it_idx][:len(prior)]
        print(f'{H:<6}' + ''.join(f'{v:>10.3f}' for v in w.tolist()))

    _, _, V = qkv_store[L]
    head_outs = []
    for H in range(n_heads):
        w  = outputs.attentions[L][0][H][it_idx]
        Vh = V[:, H*d_head:(H+1)*d_head]
        head_outs.append((w.unsqueeze(1) * Vh).sum(0))

    # ---- show the formula with real numbers for head 0 ----
    w0  = outputs.attentions[L][0][0][it_idx]
    V0  = V[:, 0:d_head]
    print(f'\n  head output = sum of (attention weight x value vector)')
    print(f'  head 0, written out:\n')
    for i in prior:
        print(f'      {w0[i]:+.3f}  x  v_{names[i]:<8} {fmt(V0[i], 3)}')
    print(f'      {"-"*54}')
    print(f'      sum     = {fmt(head_outs[0], 3)}   ({d_head} numbers)')

    z = torch.cat(head_outs)
    print(f'\n  all {n_heads} heads flattened:')
    print(f'      z = {fmt_ends(z)}')

    Wo = blocks[L].attn.c_proj.weight
    bo = blocks[L].attn.c_proj.bias

    print(f'\n  W^O  — one per layer, shared by all {n_heads} heads   (top-left 4x4 of 768x768)')
    for r in range(4):
        print('      ' + ''.join(f'{Wo[r, c]:>9.3f}' for c in range(4)))

    attn_out = z @ Wo + bo
    print(f'\n  z @ W^O  — each output slot is z dotted with one column of W^O')
    terms = ' + '.join(f'({z[k]:+.3f})({Wo[k,0]:+.3f})' for k in range(3))
    print(f'      slot 0 = {terms} + ... + b')
    print(f'             = {attn_out[0]:+.3f}')
    print(f'      attention output = {fmt(attn_out)}')

    x_old = hs[L][0, it_idx]
    print(f'\n  new x_it = old x_it + attention output')
    print(f'      old x_it     {fmt(x_old)}')
    print(f'    + attn output  {fmt(attn_out)}')
    print(f'    ' + '-'*46)
    print(f'    = new x_it     {fmt(x_old + attn_out)}')

Starting vector for "it"  (token embedding + positional encoding)
  x_it = [+0.025, -0.048, +0.144, +0.046, -0.066, -0.020, -0.252, +0.053, +0.045, +0.048, ...]   (768 numbers total)

  note: each layer also has an MLP sublayer that adds to x_it.
        the tables below trace the attention half only.


LAYER 0
Head         The       dog       did       not     cross       the      road   because        it
------------------------------------------------------------------------------------------------
0          0.336     0.158     0.080     0.094     0.092     0.043     0.048     0.087     0.061
1          0.008     0.001     0.001     0.006     0.000     0.018     0.000     0.002     0.964
2          0.286     0.074     0.169     0.069     0.026     0.052     0.067     0.201     0.054
3          0.007     0.001     0.001     0.002     0.003     0.013     0.010     0.031     0.932
4          0.089     0.021     0.064     0.054     0.072     0.047     0.157     0.267     0.230
5       

## The Calculation Slide

In [15]:
import torch, math

L, H   = 4, 3
d_head = model.config.n_embd // model.config.n_head
Q, K, _ = qkv_store[L]
s, e   = H*d_head, (H+1)*d_head

q_it   = Q[it_idx,   s:e]
k_dog  = K[dog_idx,  s:e]
k_road = K[road_idx, s:e]

N = 5

def row(name, v):
    print(f'  {name:<9} = [{", ".join(f"{x:+.3f}" for x in v[:N].tolist())}, ... ]   ({len(v)} numbers)')

print(f'GPT-2  layer {L}, head {H}\n')
row('q_it',   q_it)
row('k_dog',  k_dog)
row('k_road', k_road)

for nm, k in [('dog', k_dog), ('road', k_road)]:
    print(f'\n  q_it · k_{nm}')
    print('    ' + ' + '.join(f'({q_it[i]:+.3f})({k[i]:+.3f})' for i in range(N)) + ' + ...')
    dot = torch.dot(q_it, k).item()
    print(f'    = {dot:+.4f}      (all {len(q_it)} terms)')

GPT-2  layer 4, head 3

  q_it      = [+0.153, +0.397, -0.096, +0.584, +0.655, ... ]   (64 numbers)
  k_dog     = [+0.996, +1.208, +2.354, +0.499, +0.937, ... ]   (64 numbers)
  k_road    = [+1.569, +1.895, +3.965, +1.487, +0.929, ... ]   (64 numbers)

  q_it · k_dog
    (+0.153)(+0.996) + (+0.397)(+1.208) + (-0.096)(+2.354) + (+0.584)(+0.499) + (+0.655)(+0.937) + ...
    = +2.3431      (all 64 terms)

  q_it · k_road
    (+0.153)(+1.569) + (+0.397)(+1.895) + (-0.096)(+3.965) + (+0.584)(+1.487) + (+0.655)(+0.929) + ...
    = -38.3041      (all 64 terms)


In [16]:
w  = outputs.attentions[L][0][H][it_idx]
V  = qkv_store[L][2]
Vh = V[:, s:e]

print(f'attention weights from "it"  (layer {L}, head {H})\n')
for i in range(it_idx + 1):
    bar = '█' * int(w[i].item() * 50)
    print(f'  {tokens[i].strip():<9} {w[i]:.4f}  {bar}')

print(f'\n  dog / road = {w[dog_idx]/w[road_idx]:.0f}x\n')

out = (w.unsqueeze(1) * Vh).sum(0)
print('new "it" vector = sum of (weight x V)\n')
for i in [dog_idx, road_idx]:
    print(f'  {w[i]:.4f} x v_{tokens[i].strip():<6} [{", ".join(f"{x:+.3f}" for x in Vh[i][:4].tolist())}, ...]')
print(f'  ... plus every other token\n')
print(f'  = [{", ".join(f"{x:+.3f}" for x in out[:5].tolist())}, ...]   ({len(out)} numbers)')

attention weights from "it"  (layer 4, head 3)

  The       0.0735  ███
  dog       0.8626  ███████████████████████████████████████████
  did       0.0220  █
  not       0.0020  
  cross     0.0102  
  the       0.0044  
  road      0.0054  
  because   0.0071  
  it        0.0128  

  dog / road = 161x

new "it" vector = sum of (weight x V)

  0.8626 x v_dog    [-0.482, +0.333, -0.126, +0.234, ...]
  0.0054 x v_road   [+1.618, +0.219, -0.885, +0.218, ...]
  ... plus every other token

  = [-0.409, +0.280, -0.085, +0.190, +1.168, ...]   (64 numbers)
